# 03 - Multi-Step Reasoning And ReAct

[ReAct](https://arxiv.org/abs/2210.03629) means Reason, Act, Observe. The model decides the next useful action, a local tool performs that action, and the observation is fed back into the next step.

In production, this pattern appears in support assistants, data assistants, [RAG systems](https://arxiv.org/abs/2005.11401), incident triage, and workflow agents. The important skill is not making the loop magical. It is making it inspectable.


All embedded figures are local raster images with their source links shown where they are introduced.


## What You Will Build

You will build a small support-triage loop that:

- asks a model for the next action as [JSON](https://www.json.org/json-en.html);
- executes only allow-listed local tools;
- stores observations after each step;
- produces a concise final answer;
- avoids asking the model to reveal private [chain-of-thought](https://arxiv.org/abs/2201.11903).


## Reasoning And Retrieval Context

Multi-step prompting often combines model reasoning with retrieved or tool-provided context. RAG is not the same as ReAct, but the diagram shows the same central idea: external information is retrieved, inserted into the model context, and used to ground the final answer.

![Retrieval-augmented generation diagram](assets/rag-diagram.png)

Source: [Wikimedia Commons - RAG diagram](https://commons.wikimedia.org/wiki/File:RAG_diagram.svg)

The visual reasoning example below is a useful reminder that modern LLM workflows increasingly combine language instructions with multimodal evidence. Even though this notebook uses text-only local tools, the same orchestration pattern appears when models inspect images, documents, search results, or application state.

![Multimodal reasoning example](assets/gemini-reasoning-example.png)

Source: [Wikimedia Commons - Gemini image understanding and reasoning](https://commons.wikimedia.org/wiki/File:Gemini_image_understanding_and_reasoning.png)


## Provider And Tool-Use References

This notebook uses a simple JSON action contract rather than provider-native tool calls so the loop is easy to inspect. Provider-native [function calling](https://docs.mistral.ai/capabilities/function_calling) follows the same idea: define tools, let the model choose a tool call, execute the tool in your code, then return the result.

References:

- [Mistral function calling](https://docs.mistral.ai/capabilities/function_calling)
- [Groq API reference](https://console.groq.com/docs/api-reference)


## Provider Helper: Imports And Environment

The provider helper is split into small pieces so each responsibility is visible: environment loading, provider configuration, client creation, and the actual chat call. It uses [python-dotenv](https://pypi.org/project/python-dotenv/) for local `.env` loading and the [OpenAI Python library](https://github.com/openai/openai-python) for OpenAI-compatible provider calls.



In [ ]:
import json
import os
from dataclasses import dataclass
from pathlib import Path
from typing import Any

from dotenv import load_dotenv
from openai import OpenAI

# Load local provider settings without failing when .env does not exist yet.
load_dotenv(dotenv_path=Path(".env"), override=False)

The configuration table maps a friendly provider name to the OpenAI-compatible endpoint, API-key environment variable, and default model.



In [ ]:
@dataclass(frozen=True)
class ProviderConfig:
    """Connection details for one OpenAI-compatible chat provider."""

    name: str
    base_url: str
    api_key_env: str
    model_env: str
    default_model: str


PROVIDERS = {
    "groq": ProviderConfig(
        name="groq",
        base_url="https://api.groq.com/openai/v1",
        api_key_env="GROQ_API_KEY",
        model_env="GROQ_MODEL",
        default_model="openai/gpt-oss-20b",
    ),
    "mistral": ProviderConfig(
        name="mistral",
        base_url="https://api.mistral.ai/v1",
        api_key_env="MISTRAL_API_KEY",
        model_env="MISTRAL_MODEL",
        default_model="mistral-small-latest",
    ),
}

The next helper chooses the provider and returns either a live client or a clear dry-run reason when no key is configured.



In [ ]:
def selected_provider() -> ProviderConfig:
    """Read the provider switch once, and fail clearly for unsupported values."""

    provider_name = os.getenv("LLM_PROVIDER", "groq").strip().lower()
    if provider_name not in PROVIDERS:
        supported = ", ".join(PROVIDERS)
        raise ValueError(f"LLM_PROVIDER must be one of: {supported}")
    return PROVIDERS[provider_name]


def client_and_model() -> tuple[OpenAI | None, str, ProviderConfig, str | None]:
    """Return a configured client, or a message explaining why dry-run mode is active."""

    provider = selected_provider()
    api_key = os.getenv(provider.api_key_env)
    model = os.getenv(provider.model_env, provider.default_model)
    if not api_key:
        reason = (
            f"Dry run: set {provider.api_key_env} in .env to call {provider.name}. "
            "The surrounding code still runs so you can inspect the workflow."
        )
        return None, model, provider, reason
    return OpenAI(api_key=api_key, base_url=provider.base_url), model, provider, None

The final helper performs the model call. In dry-run mode it returns deterministic text or JSON so the notebook remains executable without credentials.



In [ ]:
def dry_chat(
    messages: list[dict[str, str]], response_format: dict[str, Any] | None = None
) -> str:
    """Small deterministic stand-in used when no API key is configured."""

    latest_user = next(
        (m["content"] for m in reversed(messages) if m["role"] == "user"), ""
    )
    if response_format and response_format.get("type") == "json_object":
        return json.dumps(
            {
                "label": "ACCOUNT_ACCESS",
                "priority": "medium",
                "summary": "Dry-run structured response for a support request.",
                "confidence": 0.72,
            },
            indent=2,
        )
    return (
        "DRY RUN RESPONSE\n"
        "- Configure .env with a provider key to replace this with a live model response.\n"
        f"- Last user message preview: {latest_user[:160]}"
    )

The live helper now has one job: choose dry-run mode or call the provider with the requested messages and optional JSON response format.



In [ ]:
def chat(
    messages: list[dict[str, str]],
    *,
    temperature: float = 0.2,
    response_format: dict[str, Any] | None = None,
) -> str:
    """Call the selected provider, falling back to dry-run text when no key exists."""

    client, model, provider, dry_reason = client_and_model()
    if client is None:
        print(dry_reason)
        return dry_chat(messages, response_format=response_format)

    # Build kwargs dynamically so response_format is sent only when requested.
    request: dict[str, Any] = {
        "model": model,
        "messages": messages,
        "temperature": temperature,
    }
    if response_format is not None:
        request["response_format"] = response_format

    response = client.chat.completions.create(**request)
    return response.choices[0].message.content or ""

## Local Tools

Tools should be boring and explicit. The model may request an action, but [Python](https://docs.python.org/3.11/) decides whether that action is allowed and how it is executed.


In [ ]:
SUPPORT_TICKETS = {
    "T-104": {
        "customer": "Acme Analytics",
        "plan": "Team",
        "issue": "MFA reset after phone replacement",
        "status": "waiting_for_identity_verification",
    },
    "T-205": {
        "customer": "Northwind Labs",
        "plan": "Enterprise",
        "issue": "CSV export returns intermittent 500 errors",
        "status": "engineering_investigating",
    },
}

POLICY_SNIPPETS = {
    "mfa_reset": "For MFA resets, verify identity with two account recovery signals before disabling MFA.",
    "bug_escalation": "Escalate repeatable 5xx errors with request IDs, timestamps, and affected workspace.",
}

The local tools below are deliberately small. The model can request a tool call, but the Python code controls what data can be accessed.



In [ ]:
from collections.abc import Callable


def lookup_ticket(ticket_id: str) -> dict[str, str]:
    """Return support-ticket details from the local demo store."""

    return SUPPORT_TICKETS.get(ticket_id, {"error": "unknown ticket"})


def lookup_policy(topic: str) -> dict[str, str]:
    """Return the operational policy text for a known support topic."""

    return {"topic": topic, "policy": POLICY_SNIPPETS.get(topic, "No policy found.")}


TOOLS: dict[str, Callable[..., dict[str, str]]] = {
    "lookup_ticket": lookup_ticket,
    "lookup_policy": lookup_policy,
}

## Action Contract

The model must choose from a constrained JSON shape. This gives the loop a stable interface even when the natural-language task varies.


In [ ]:
ACTION_SCHEMA = {
    "type": "object",
    "properties": {
        "action": {
            "type": "string",
            "enum": ["lookup_ticket", "lookup_policy", "final_answer"],
        },
        "arguments": {"type": "object"},
        "final_answer": {"type": ["string", "null"]},
    },
    "required": ["action", "arguments", "final_answer"],
    "additionalProperties": False,
}

SYSTEM_PROMPT = f"""You help a support operator decide the next step.

Return only JSON matching this schema:
{json.dumps(ACTION_SCHEMA, indent=2)}

Rules:
- Use lookup_ticket when you need ticket details.
- Use lookup_policy when you need the policy for mfa_reset or bug_escalation.
- Use final_answer only when you have enough information.
- Do not reveal hidden chain-of-thought; keep final answers concise."""

In [ ]:
def dry_react_action(
    task: str, observations: list[dict[str, object]]
) -> dict[str, object]:
    """Deterministic fallback so the loop is runnable without an API key."""

    if not observations:
        return {
            "action": "lookup_ticket",
            "arguments": {"ticket_id": "T-104"},
            "final_answer": None,
        }
    if len(observations) == 1:
        return {
            "action": "lookup_policy",
            "arguments": {"topic": "mfa_reset"},
            "final_answer": None,
        }
    return {
        "action": "final_answer",
        "arguments": {},
        "final_answer": "Verify two recovery signals before disabling MFA for ticket T-104.",
    }

## Ask The Model For The Next Action

The prompt includes the task and the observations collected so far. Notice that observations are data, not instructions. The system message still controls the action contract.


In [ ]:
def next_action(task: str, observations: list[dict[str, object]]) -> dict[str, object]:
    """Request the next action from the model and parse it as JSON."""

    client, _, _, dry_reason = client_and_model()
    if client is None:
        print(dry_reason)
        return dry_react_action(task, observations)

    user_prompt = f"""Task: {task}

Observations so far:
{json.dumps(observations, indent=2)}"""
    raw = chat(
        [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": user_prompt},
        ],
        temperature=0.1,
        response_format={"type": "json_object"},
    )
    return json.loads(raw)

## Execute Only Allow-Listed Tools

This is the security boundary. The model can request `lookup_ticket`, but it cannot execute arbitrary Python or invent new tool names.


In [ ]:
from typing import Any


def execute_action(action: dict[str, object]) -> dict[str, object]:
    """Run an allow-listed tool call and return an observation record."""

    action_name = str(action.get("action"))
    if action_name == "final_answer":
        return {"type": "final", "content": action.get("final_answer")}

    if action_name not in TOOLS:
        return {"type": "error", "content": f"Blocked unknown tool: {action_name}"}

    raw_arguments = action.get("arguments") or {}
    if not isinstance(raw_arguments, dict):
        return {"type": "error", "content": "Tool arguments must be an object."}

    # Convert JSON values to strings because the two demo tools accept string arguments.
    arguments = {str(key): str(value) for key, value in raw_arguments.items()}
    tool = TOOLS[action_name]
    result = tool(**arguments)
    return {
        "type": "tool_result",
        "tool": action_name,
        "arguments": arguments,
        "content": result,
    }

## Run The Loop

The loop stops when the model returns `final_answer` or when it reaches the step limit. The step limit protects you from infinite loops and runaway API usage.


In [ ]:
def run_react(task: str, max_steps: int = 4) -> dict[str, Any]:
    """Run a small ReAct loop and keep the trace inspectable."""

    observations: list[dict[str, Any]] = []
    for step in range(1, max_steps + 1):
        action = next_action(task, observations)
        observation = execute_action(action)
        observations.append(
            {"step": step, "action": action, "observation": observation}
        )
        if observation["type"] == "final":
            break
    return {
        "task": task,
        "trace": observations,
        "final": observations[-1]["observation"]["content"],
    }


result = run_react("For ticket T-104, decide the next safe support action.")
print(json.dumps(result, indent=2))

## Final Answer Discipline

For multi-step work, keep two outputs separate:

- the trace, which is for developers and auditors;
- the final answer, which is for the end user or operator.

The final answer should cite observations or tool results, not hidden reasoning.


In [ ]:
print("Final answer:")
print(result["final"])

print("\nTrace summary:")
for item in result["trace"]:
    action = item["action"]["action"]
    observation_type = item["observation"]["type"]
    print(f"Step {item['step']}: {action} -> {observation_type}")

## Further Reading

- [ReAct: Synergizing Reasoning and Acting in Language Models](https://arxiv.org/abs/2210.03629)
- [Retrieval-Augmented Generation for Knowledge-Intensive NLP Tasks](https://arxiv.org/abs/2005.11401)
- [Mistral function calling](https://docs.mistral.ai/capabilities/function_calling)
- [LangChain tool calling](https://python.langchain.com/docs/concepts/tool_calling/)
- [OpenAI function calling](https://platform.openai.com/docs/guides/function-calling)



## Exercise

Add a new tool named `lookup_customer_plan(customer: str)`. Then update the action prompt so the model can use it before producing a final answer.

Keep the allow-list boundary intact: the tool name must appear in `TOOLS`, arguments must be a JSON object, and the loop should still stop after `max_steps`.


## Correction

A tool becomes available to the model only after two changes: implement and allow-list the function, then advertise the new action in the JSON contract and system prompt. The execution boundary remains unchanged because `execute_action` still rejects names outside `TOOLS`.


In [ ]:
CUSTOMER_PLANS = {
    "Acme Analytics": {
        "plan": "Team",
        "mfa_recovery": "identity verification required",
    },
}


def lookup_customer_plan(customer: str) -> dict[str, str]:
    """Return plan-level support context for an allow-listed customer."""

    return CUSTOMER_PLANS.get(customer, {"error": "unknown customer"})


TOOLS["lookup_customer_plan"] = lookup_customer_plan
print(
    execute_action(
        {"action": "lookup_customer_plan", "arguments": {"customer": "Acme Analytics"}}
    )
)

Now extend only the action contract. In a live call, the model can select `lookup_customer_plan`; the same allow-listed dispatcher executes it and the existing `max_steps` argument still limits the loop.


In [ ]:
extended_schema = {
    **ACTION_SCHEMA,
    "properties": {
        **ACTION_SCHEMA["properties"],
        "action": {
            "type": "string",
            "enum": [
                "lookup_ticket",
                "lookup_policy",
                "lookup_customer_plan",
                "final_answer",
            ],
        },
    },
}
SYSTEM_PROMPT = f"""You help a support operator decide the next step.

Return only JSON matching this schema:
{json.dumps(extended_schema, indent=2)}

Use lookup_customer_plan when plan context affects the safe next action.
Use final_answer only after gathering enough information."""
print(SYSTEM_PROMPT)